# ПР-3. Пайплайн предобработки
Цель: собрать пайплайн со своим трансформером, который не допускает утечки данных из test в train.

In [1]:
import sys
sys.path.append("..")   # чтобы увидеть папку src

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from src.transformers import CreditCleaner

df = pd.read_csv("../data/cs-training.csv", index_col=0)
df = df.rename(columns={
    "SeriousDlqin2yrs": "default",
    "RevolvingUtilizationOfUnsecuredLines": "utilization",
    "NumberOfTime30-59DaysPastDueNotWorse": "late_30_59",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "income",
    "NumberOfOpenCreditLinesAndLoans": "open_credits",
    "NumberOfTimes90DaysLate": "late_90",
    "NumberRealEstateLoansOrLines": "real_estate",
    "NumberOfTime60-89DaysPastDueNotWorse": "late_60_89",
    "NumberOfDependents": "dependents",
})
df.shape

(150000, 11)

In [2]:
X = df.drop(columns="default")
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print("Доля просрочек в train:", y_train.mean(), "в test:", y_test.mean())

(120000, 10) (30000, 10)
Доля просрочек в train: 0.06684166666666666 в test: 0.06683333333333333


## Стратегия обработки
- Служебные коды 96/98 и возраст < 18: заменяются на пропуск (невозможные значения).
- Выбросы utilization, debt_ratio, income: обрезаются сверху по 99-му процентилю, вычисленному на train.
- Пропуски (income, dependents, а также новые от замены кодов): заполняются медианой, а не средним, потому что распределения скошены и среднее сильно зависит от выбросов.
- Масштабирование: StandardScaler, чтобы признаки с большими числами (доход) не подавляли признаки с маленькими (возраст, число просрочек).
- Новые признаки: total_late (общее число просрочек) и income_missing (признак «доход неизвестен»).

In [3]:
pipe = Pipeline([
    ("cleaner", CreditCleaner()),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# fit_transform на train: пайплайн ОБУЧАЕТСЯ и преобразует
X_train_p = pipe.fit_transform(X_train)
# transform на test: только ПРИМЕНЯЕТ, не переобучается
X_test_p = pipe.transform(X_test)

print("train:", X_train_p.shape)
print("test: ", X_test_p.shape)

train: (120000, 12)
test:  (30000, 12)


In [4]:
print("Границы, выученные на train:", pipe.named_steps["cleaner"].caps_)
print("Если бы считали по всем данным:", {c: df[c].quantile(0.99) for c in ["utilization", "debt_ratio", "income"]})
print("Медианы для заполнения пропусков (train):", pipe.named_steps["imputer"].statistics_)

Границы, выученные на train: {'utilization': np.float64(1.0944386634299974), 'debt_ratio': np.float64(4977.0), 'income': np.float64(25000.0)}
Если бы считали по всем данным: {'utilization': np.float64(1.0929557681400022), 'debt_ratio': np.float64(4979.040000000037), 'income': np.float64(25000.0)}
Медианы для заполнения пропусков (train): [1.5331848e-01 5.2000000e+01 0.0000000e+00 3.6619425e-01 5.3900000e+03
 8.0000000e+00 0.0000000e+00 1.0000000e+00 0.0000000e+00 0.0000000e+00
 0.0000000e+00 0.0000000e+00]


## Проверка на отсутствие утечки
Верхние границы для обрезки выбросов, вычисленные только по обучающей выборке, немного отличаются от границ, вычисленных по всем данным: utilization 1,0944 против 1,0930, debt_ratio 4977 против 4979. Это показывает, что трансформер использует только train, а тестовые данные на выученные границы не влияют. Для income границы совпали (25 000), так как в данных много одинаковых круглых значений дохода.
Медианы для заполнения пропусков тоже выучены на train (например, медианный доход 5 390, медианный возраст 52 года) и затем применяются к test без пересчёта. Число столбцов в train и test одинаковое (12), значит, пайплайн применяется к обеим частям одинаково.

In [5]:
# искусственно увеличиваем train в 10 раз (120 тыс. -> 1,2 млн строк)
big = pd.concat([X_train] * 10, ignore_index=True)
print("Размер увеличенных данных:", big.shape)

pipe_big = Pipeline([
    ("cleaner", CreditCleaner()),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

data = big
for name, step in pipe_big.steps:
    t = time.perf_counter()
    data = step.fit_transform(data)
    print(f"{name}: {time.perf_counter() - t:.2f} сек")

Размер увеличенных данных: (1200000, 10)
cleaner: 0.89 сек
imputer: 4.09 сек
scaler: 0.53 сек


## Проверка производительности
Обучающая выборка искусственно увеличена в 10 раз (до 1,2 млн строк, 10 признаков). Время выполнения шагов пайплайна (fit_transform):
| Шаг | Время, сек |
|---|---|
| cleaner (собственный трансформер) | 0,74 |
| imputer (медианное заполнение) | 3,98 |
| scaler (StandardScaler) | 0,47 |

**Узкое место:** SimpleImputer с медианой, на него приходится около 77% общего времени (около 5,2 сек). Вероятная причина: для вычисления медианы нужно упорядочить значения в каждом столбце, что дороже, чем вычисление среднего и разброса в StandardScaler. Собственный трансформер работает быстро (0,74 сек), поэтому он не является проблемой. На исходных 120 тысячах строк время шагов в десять раз меньше и незаметно.

## Тесты трансформера
Файл с тестами: `tests/test_transformers.py`. Результат запуска `python -m pytest -v`:
```
============================================ test session starts ===========================================

platform win32 -- Python 3.12.7, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\user\.vscode\MLSU\credit-risk-ml\venv\scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\user\.vscode\MLSU\credit-risk-ml
plugins: platformdirs-4.13.0
collected 3 items                                                                                                                          

tests/test_transformers.py::test_caps_learned_on_train_applied_to_new_data PASSED                                                    [33%]
tests/test_transformers.py::test_service_codes_become_nan PASSED                                                                     [66%]
tests/test_transformers.py::test_input_is_not_modified PASSED                                                                        [100%]

============================================= 3 passed in 5.14s ============================================
```
Тесты проверяют: (1) границы выучены на train и применяются к новым данным без изменений, (2) служебные коды 96/98 превращаются в пропуски, (3) исходная таблица не изменяется.